In [0]:
-- Record current counts before pausing
SELECT
  (SELECT COUNT(*) FROM wiki_poc.poc.bronze_recentchange_raw)     AS bronze_before,
  (SELECT COUNT(*) FROM wiki_poc.poc.silver_recentchange_enwiki)  AS silver_before;

Pause the DLT pipeline

Pipelines → your pipeline → Stop (not delete). Fargate keeps running
and writing JSONL files to S3 throughout.

Wait 30 minutes.

Restart the pipeline

Pipelines → your pipeline → Start.

Watch the pipeline event log — it should show Auto Loader catching up on
the accumulated S3 files. The Bronze row count should climb rapidly until
it catches up, then settle back to the normal ~25–40 events/sec rate.

In [0]:
-- Confirm full catch-up
-- Run ~5 minutes after restart
SELECT
  (SELECT COUNT(*) FROM wiki_poc.poc.bronze_recentchange_raw)     AS bronze_after,
  (SELECT COUNT(*) FROM wiki_poc.poc.silver_recentchange_enwiki)  AS silver_after;

In [0]:
-- Confirm no gap in Silver's 5-minute buckets across the pause window
SELECT
  from_unixtime(FLOOR(unix_timestamp(ingest_timestamp) / 300) * 300) AS bucket_start,
  COUNT(*) AS silver_events
FROM wiki_poc.poc.silver_recentchange_enwiki
WHERE ingest_timestamp >= NOW() - INTERVAL 45 MINUTES
GROUP BY 1
ORDER BY 1;